# Project : Patient Readmission Risk Prediction

## Notebook 2: Preprocessing and Feature Engineering

This notebook prepares the dataset for modelling.

The main goal here is to convert the raw data into a cleaner, model-ready version.

In this notebook, I will:
- reload the raw dataset
- clean the column names
- remove unnecessary identifier columns
- inspect and convert date columns carefully
- prepare the dataset for later encoding and modelling

This stage is important because even if a dataset has no missing values, it can still have formatting issues, leakage risks, and columns that should not be used in prediction.

In [186]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## Load the raw dataset again

I am starting from the raw file so that the preprocessing steps are fully reproducible in one notebook.

In [187]:
df = pd.read_csv("../data/raw/Healthcare Data Analysis for readmission.csv")
print("Dataset loaded successfully.")

Dataset loaded successfully.


In [188]:
df.head()

,hospital_name,Admission_date,hospital_id,hospital_beds_available,occupied_beds,hospital_ward,patient_id,patient_gender,patient_age,patient_race,patient_sat_score,patient_first_initial,patient_last_name,patient_waittime,department_referral,time_slot,doctor_id,doctor_name,doctor_specialty,patient_assigned_doctor,patient_checkin_date,patient_checkout_date,patient_disease,patient_length_of_stay,discharge_status,readmission
0,The Johns Hopkins Hospital,30-01-2024,3946,260,90,Maternity,1421,Female,48,Asian,490,t,Rogers,120,Urology or Nephrology,02:34:00 PM,1725,Justin Morris,Rheumatology,False,14-07-2024,19-07-2024,Anxiety Disorders,6,Deceased,1
1,The Johns Hopkins Hospital,20-03-2022,7147,250,90,ICU,4922,Male,40,Black,1210,i,Spencer,60,Neurology,04:07:00 PM,7510,Latoya Moss,Pulmonology or Allergy and Immunology,False,19-07-2024,07/07/2024,Urinary Tract Infection (UTI),2,Deceased,1
2,The Johns Hopkins Hospital,13-07-2021,4174,350,220,Pediatrics,9804,Female,74,Black,920,Y,Hall,120,Neurology,07:16:00 PM,7137,Michael Allen,Neurology,False,20-07-2024,29-06-2024,Hypertension (High Blood Pressure),17,Recovered,0
3,The Johns Hopkins Hospital,26-01-2021,2466,190,350,Pediatrics,5622,Male,82,Hispanic,1010,s,Rodriguez,75,Gastroenterology,07:27:00 AM,8801,Rebecca Mccullough,Cardiology,False,25-06-2024,16-07-2024,Arthritis,28,Recovered,0
4,The Johns Hopkins Hospital,06/02/2024,3014,220,390,Surgery,3314,Male,62,Hispanic,570,o,Duffy,90,Dermatology,01:54:00 PM,3588,John Giles,Rheumatology,False,21-07-2024,19-07-2024,Cancer,16,Transferred,1


## Standardize the column names

To make the preprocessing easier, I will convert all column names to lowercase and replace spaces with underscores.

In [189]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")
print(df.columns.tolist())

['hospital_name', 'admission_date', 'hospital_id', 'hospital_beds_available', 'occupied_beds', 'hospital_ward', 'patient_id', 'patient_gender', 'patient_age', 'patient_race', 'patient_sat_score', 'patient_first_initial', 'patient_last_name', 'patient_waittime', 'department_referral', 'time_slot', 'doctor_id', 'doctor_name', 'doctor_specialty', 'patient_assigned_doctor', 'patient_checkin_date', 'patient_checkout_date', 'patient_disease', 'patient_length_of_stay', 'discharge_status', 'readmission']


## Drop obvious identifier and personal detail columns

Some columns should not be used in modelling because they are identifiers, names, or personal text fields that do not provide meaningful predictive value.

Dropping them early also makes the rest of the workflow easier to manage.

In [190]:
cols_to_drop = [
    "hospital_name",
    "hospital_id",
    "patient_id",
    "patient_first_initial",
    "patient_last_name",
    "doctor_id",
    "doctor_name"
]

df = df.drop(columns=cols_to_drop)
print("Dropped columns:")
print(cols_to_drop)

Dropped columns:
['hospital_name', 'hospital_id', 'patient_id', 'patient_first_initial', 'patient_last_name', 'doctor_id', 'doctor_name']


In [191]:
print("Current shape after dropping identifier columns:", df.shape)
print(df.columns.tolist())

Current shape after dropping identifier columns: (10000, 19)
['admission_date', 'hospital_beds_available', 'occupied_beds', 'hospital_ward', 'patient_gender', 'patient_age', 'patient_race', 'patient_sat_score', 'patient_waittime', 'department_referral', 'time_slot', 'doctor_specialty', 'patient_assigned_doctor', 'patient_checkin_date', 'patient_checkout_date', 'patient_disease', 'patient_length_of_stay', 'discharge_status', 'readmission']


## Inspect the date columns more carefully

From the first notebook, I already noticed that the date columns may contain mixed formats.

Before converting them, I want to inspect a few sample values again and then convert them carefully.

In [192]:
date_cols = ["admission_date", "patient_checkin_date", "patient_checkout_date"]

for col in date_cols:
    print(f"{col} sample values:")
    print(df[col].head(10))
    print("-" * 50)

admission_date sample values:
0    30-01-2024
1    20-03-2022
2    13-07-2021
3    26-01-2021
4    06/02/2024
5    29-04-2022
6    25-07-2020
7    18-05-2022
8    20-07-2022
9    29-05-2022
Name: admission_date, dtype: object
--------------------------------------------------
patient_checkin_date sample values:
0    14-07-2024
1    19-07-2024
2    20-07-2024
3    25-06-2024
4    21-07-2024
5    03/07/2024
6    21-07-2024
7    19-07-2024
8    09/07/2024
9    14-07-2024
Name: patient_checkin_date, dtype: object
--------------------------------------------------
patient_checkout_date sample values:
0    19-07-2024
1    07/07/2024
2    29-06-2024
3    16-07-2024
4    19-07-2024
5    25-06-2024
6    16-07-2024
7    14-07-2024
8    18-07-2024
9    12/07/2024
Name: patient_checkout_date, dtype: object
--------------------------------------------------


## Convert date columns

Since the date formats appear mixed, I will use `dayfirst=True` and `errors="coerce"`.

This helps pandas interpret values like `30-01-2024` correctly and avoids crashing if any value cannot be parsed.

In [193]:
# First normalize separators so all dates use the same symbol
for col in date_cols:
    df[col] = df[col].astype(str).str.replace("/", "-", regex=False)

# Convert carefully
for col in date_cols:
    df[col] = pd.to_datetime(
        df[col],
        format="mixed",
        dayfirst=True,
        errors="coerce"
    )

print("Date conversion completed with mixed-format handling.")

Date conversion completed with mixed-format handling.


In [194]:
df[date_cols].head()

,admission_date,patient_checkin_date,patient_checkout_date
0,2024-01-30,2024-07-14,2024-07-19
1,2022-03-20,2024-07-19,2024-07-07
2,2021-07-13,2024-07-20,2024-06-29
3,2021-01-26,2024-06-25,2024-07-16
4,2024-02-06,2024-07-21,2024-07-19


## Check for any remaining missing values in the date columns

After fixing the mixed date formats, I want to confirm whether any date values still failed to convert.

If there are still missing values, I need to decide whether they are genuine missing records or conversion issues.

In [195]:
print(df[date_cols].isnull().sum())

admission_date           0
patient_checkin_date     0
patient_checkout_date    0
dtype: int64


## Create date-based features

Raw dates are usually not ideal for machine learning models.

Instead, I will create simpler time-related features that may capture useful patterns such as seasonality and weekday effects.

In [196]:
df["admission_month"] = df["admission_date"].dt.month
df["admission_dayofweek"] = df["admission_date"].dt.dayofweek
df["checkin_month"] = df["patient_checkin_date"].dt.month
df["checkout_month"] = df["patient_checkout_date"].dt.month

print("Date-based features created successfully.")

Date-based features created successfully.


In [197]:
df[[
    "admission_date",
    "admission_month",
    "admission_dayofweek",
    "patient_checkin_date",
    "checkin_month",
    "patient_checkout_date",
    "checkout_month"
]].head()

,admission_date,admission_month,admission_dayofweek,patient_checkin_date,checkin_month,patient_checkout_date,checkout_month
0,2024-01-30,1,1,2024-07-14,7,2024-07-19,7
1,2022-03-20,3,6,2024-07-19,7,2024-07-07,7
2,2021-07-13,7,1,2024-07-20,7,2024-06-29,6
3,2021-01-26,1,1,2024-06-25,6,2024-07-16,7
4,2024-02-06,2,1,2024-07-21,7,2024-07-19,7


## Create stay gap features

The time between admission, check-in, and checkout may provide additional operational signals.

These features may help the model capture patient flow and discharge timing patterns.

In [198]:
df["days_from_admission_to_checkin"] = (
    df["patient_checkin_date"] - df["admission_date"]
).dt.days

df["days_from_checkin_to_checkout"] = (
    df["patient_checkout_date"] - df["patient_checkin_date"]
).dt.days

print("Gap features created successfully.")

Gap features created successfully.


## Create cleaned duration features

The raw date differences reveal that some records have inconsistent temporal ordering, which leads to negative values.

To make the features more stable for modelling, I will use absolute day differences as operational duration features.

In [199]:
df["days_from_admission_to_checkin"] = (
    df["patient_checkin_date"] - df["admission_date"]
).dt.days.abs()

df["days_from_checkin_to_checkout"] = (
    df["patient_checkout_date"] - df["patient_checkin_date"]
).dt.days.abs()

print("Cleaned duration features created successfully.")

Cleaned duration features created successfully.


In [200]:
df[[
    "days_from_admission_to_checkin",
    "days_from_checkin_to_checkout"
]].head()

,days_from_admission_to_checkin,days_from_checkin_to_checkout
0,166,5
1,852,12
2,1103,21
3,1246,21
4,166,2


## Review the remaining categorical columns

Before encoding the categorical variables, I want to inspect which object-type columns are still present in the dataset.

This helps me understand:
- which columns are meaningful categories
- which columns may behave like identifiers
- which columns may need to be dropped before modelling

In [201]:
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Current categorical columns:")
print(categorical_cols)

Current categorical columns:
['hospital_ward', 'patient_gender', 'patient_race', 'department_referral', 'time_slot', 'doctor_specialty', 'patient_disease', 'discharge_status']


## Check how many unique values each categorical column has

This is useful because columns with very high uniqueness may not be good candidates for standard encoding.

In [202]:
for col in categorical_cols:
    print(f"{col}: {df[col].nunique()} unique values")

hospital_ward: 5 unique values
patient_gender: 2 unique values
patient_race: 5 unique values
department_referral: 19 unique values
time_slot: 1439 unique values
doctor_specialty: 19 unique values
patient_disease: 28 unique values
discharge_status: 3 unique values


## Process the time_slot feature

The raw `time_slot` column has too many unique values and behaves more like a time variable than a category.

Instead of using it directly, I will extract the hour of the day.

This keeps the time information while reducing unnecessary dimensionality.


## Inspect raw time_slot values

Before converting the time values, I need to inspect the actual raw format used in the dataset.

In [203]:
print(df["time_slot"].astype(str).head(20))

0     02:34:00 PM
1     04:07:00 PM
2     07:16:00 PM
3     07:27:00 AM
4     01:54:00 PM
5     10:35:00 AM
6     03:48:00 PM
7     11:35:00 PM
8     01:02:00 PM
9     12:36:00 PM
10    08:22:00 AM
11    05:57:00 AM
12    06:29:00 PM
13    03:50:00 AM
14    05:10:00 PM
15    03:59:00 PM
16    11:17:00 AM
17    10:48:00 PM
18    11:09:00 AM
19    06:48:00 PM
Name: time_slot, dtype: object


## Convert the time_slot column correctly

The raw time values use a 12-hour format with seconds and an AM/PM indicator.

I will now convert them properly and extract the hour of the day.

In [204]:
df["time_slot"] = pd.to_datetime(
    df["time_slot"],
    format="%I:%M:%S %p",
    errors="coerce"
)

print("time_slot converted successfully.")

time_slot converted successfully.


In [205]:
df["hour_of_day"] = df["time_slot"].dt.hour

df[["time_slot", "hour_of_day"]].head()

,time_slot,hour_of_day
0,1900-01-01 14:34:00,14
1,1900-01-01 16:07:00,16
2,1900-01-01 19:16:00,19
3,1900-01-01 07:27:00,7
4,1900-01-01 13:54:00,13


In [206]:
print("Unique hour values:", df["hour_of_day"].nunique())
print(sorted(df["hour_of_day"].dropna().unique()))

Unique hour values: 24
[np.int32(0), np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12), np.int32(13), np.int32(14), np.int32(15), np.int32(16), np.int32(17), np.int32(18), np.int32(19), np.int32(20), np.int32(21), np.int32(22), np.int32(23)]


## Drop the original time_slot column

Now that the useful hour-based feature has been created, the original raw time field is no longer required.

In [207]:
df = df.drop(columns=["time_slot"])

print("time_slot dropped successfully.")
print("Current shape:", df.shape)

time_slot dropped successfully.
Current shape: (10000, 25)


## Prepare the feature and target split

At this stage, the target variable is separated from the feature set.

The target for this project is whether the patient was readmitted.

In [208]:
df = df.drop(columns=["patient_assigned_doctor"])
print("Dropped patient_assigned_doctor.")

Dropped patient_assigned_doctor.


In [209]:
X = df.drop(columns=["readmission"])
y = df["readmission"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print(y.value_counts())

Feature shape: (10000, 23)
Target shape: (10000,)
readmission
0    5858
1    4142
Name: count, dtype: int64


## Identify categorical columns for encoding

Before modelling, categorical columns need to be converted into numeric form.

I will use one-hot encoding for the remaining categorical features.

In [210]:
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

print("Categorical columns to encode:")
print(categorical_cols)

Categorical columns to encode:
['hospital_ward', 'patient_gender', 'patient_race', 'department_referral', 'doctor_specialty', 'patient_disease', 'discharge_status']


In [211]:
X_encoded = pd.get_dummies(
    X,
    columns=categorical_cols,
    drop_first=True
)

print("Encoding completed successfully.")

Encoding completed successfully.


In [212]:
print("Encoded feature shape:", X_encoded.shape)
X_encoded.head()

Encoded feature shape: (10000, 90)


,admission_date,hospital_beds_available,occupied_beds,patient_age,patient_sat_score,patient_waittime,patient_checkin_date,patient_checkout_date,patient_length_of_stay,admission_month,admission_dayofweek,checkin_month,checkout_month,days_from_admission_to_checkin,days_from_checkin_to_checkout,hour_of_day,hospital_ward_ICU,hospital_ward_Maternity,hospital_ward_Pediatrics,hospital_ward_Surgery,patient_gender_Male,patient_race_Black,patient_race_Hispanic,patient_race_Other,patient_race_White,department_referral_Cardiology,department_referral_Cardiology or Internal Medicine,department_referral_Dermatology,department_referral_Endocrinology,department_referral_Gastroenterology,department_referral_Hepatology,department_referral_Hepatology or Gastroenterology,department_referral_Infectious Disease,department_referral_Infectious Disease or Internal Medicine,department_referral_Nephrology,department_referral_Neurology,department_referral_Oncology,department_referral_Psychiatry,department_referral_Pulmonology,department_referral_Pulmonology or Allergy and Immunology,department_referral_Pulmonology or Infectious Disease,department_referral_Rheumatology,department_referral_Urology or Nephrology,doctor_specialty_Cardiology,doctor_specialty_Cardiology or Internal Medicine,doctor_specialty_Dermatology,doctor_specialty_Endocrinology,doctor_specialty_Gastroenterology,doctor_specialty_Hepatology,doctor_specialty_Hepatology or Gastroenterology,doctor_specialty_Infectious Disease,doctor_specialty_Infectious Disease or Internal Medicine,doctor_specialty_Nephrology,doctor_specialty_Neurology,doctor_specialty_Oncology,doctor_specialty_Psychiatry,doctor_specialty_Pulmonology,doctor_specialty_Pulmonology or Allergy and Immunology,doctor_specialty_Pulmonology or Infectious Disease,doctor_specialty_Rheumatology,doctor_specialty_Urology or Nephrology,patient_disease_Alzheimer’s Disease,patient_disease_Anxiety Disorders,patient_disease_Arthritis,patient_disease_Asthma,patient_disease_Autoimmune Diseases,patient_disease_COVID-19,patient_disease_Cancer,patient_disease_Celiac Disease,patient_disease_Chronic Kidney Disease,patient_disease_Chronic Obstructive Pulmonary Disease (COPD),patient_disease_Depression,patient_disease_Diabetes,patient_disease_Epilepsy,patient_disease_Gastroesophageal Reflux Disease (GERD),patient_disease_HIV/AIDS,patient_disease_Heart Disease,patient_disease_Hepatitis,patient_disease_Hypertension (High Blood Pressure),patient_disease_Influenza (Flu),patient_disease_Irritable Bowel Syndrome (IBS),patient_disease_Liver Cirrhosis,patient_disease_Migraine,patient_disease_Parkinson’s Disease,patient_disease_Pneumonia,patient_disease_Skin Diseases,patient_disease_Stroke,patient_disease_Urinary Tract Infection (UTI),discharge_status_Recovered,discharge_status_Transferred
0,2024-01-30,260,90,48,490,120,2024-07-14,2024-07-19,6,1,1,7,7,166,5,14,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False
1,2022-03-20,250,90,40,1210,60,2024-07-19,2024-07-07,2,3,6,7,7,852,12,16,True,False,False,False,True,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False
2,2021-07-13,350,220,74,920,120,2024-07-20,2024-06-29,17,7,1,7,6,1103,21,19,False,False,True,False,False,True,False,False,False,False,False,False,False,False,False,False,Fal

## Check for any remaining non-numeric columns

After encoding, all features should now be numeric.

In [213]:
print(X_encoded.dtypes.value_counts())

bool              74
int64              8
int32              5
datetime64[ns]     3
Name: count, dtype: int64


## Remove raw datetime columns

The useful time-based features have already been engineered.

The original raw datetime columns are no longer needed for modelling and should be removed.

In [214]:
datetime_cols = X_encoded.select_dtypes(include=["datetime64[ns]"]).columns.tolist()

print("Datetime columns to drop:")
print(datetime_cols)

Datetime columns to drop:
['admission_date', 'patient_checkin_date', 'patient_checkout_date']


In [215]:
X_encoded = X_encoded.drop(columns=datetime_cols)

print("Datetime columns dropped successfully.")
print("Updated shape:", X_encoded.shape)

Datetime columns dropped successfully.
Updated shape: (10000, 87)


In [216]:
print(X_encoded.dtypes.value_counts())

bool     74
int64     8
int32     5
Name: count, dtype: int64


## Convert boolean features to integer

The one-hot encoded columns are currently stored as boolean values.

I will convert them into 0/1 integer format to keep the feature matrix fully numeric and model-friendly.

In [217]:
bool_cols = X_encoded.select_dtypes(include=["bool"]).columns

X_encoded[bool_cols] = X_encoded[bool_cols].astype(int)

print("Boolean columns converted to integer successfully.")

Boolean columns converted to integer successfully.


In [218]:
print(X_encoded.dtypes.value_counts())
print("Final encoded feature shape:", X_encoded.shape)

int64    82
int32     5
Name: count, dtype: int64
Final encoded feature shape: (10000, 87)


## Summary of preprocessing

In this notebook, I:
- removed identifier columns
- fixed mixed date formats
- engineered time-based features
- created duration-based operational features
- converted time slots into hourly features
- one-hot encoded categorical variables
- removed raw datetime columns
- prepared a fully numeric feature matrix for modelling

The dataset is now ready for train-test splitting and model training in the next notebook.